# ERA5 vs GFS NWP — weather skill baseline

Compare **ERA5 reanalysis** (hindcast truth) with **GFS operational forecasts** at the study plant point (**52.5°N, 13.4°E** — same as ERA5 downloads).

1. Plot one week of ERA5 vs NWP for temperature, 10 m wind, and downward shortwave.
2. Compute **lead-time MAE** of NWP against ERA5 — a raw **weather skill** baseline before any generation model.

**Prerequisites**

- Cached ERA5 monthly file: `data/raw/era5_2019_de_bbox5deg_06.nc` (see `py scripts/download_era5.py`).
- Network access for Herbie / NOAA GFS open data (first run downloads GRIB subsets; expect several minutes).

See [docs/data_sources.md](../docs/data_sources.md) and [docs/methodology.md](../docs/methodology.md).

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from energy_features.weather import (
    DEFAULT_GFS_LEAD_HOURS,
    DEFAULT_LATITUDE,
    DEFAULT_LONGITUDE,
    ERA5Loader,
    extract_nwp_forecast_point,
)

TZ = "UTC"
LAT = DEFAULT_LATITUDE
LON = DEFAULT_LONGITUDE
WEEK_START = "2019-06-03"
WEEK_END = "2019-06-09"
LEADS = list(DEFAULT_GFS_LEAD_HOURS)
COMPARE_VARS = ("t2m_c", "wind_speed_10m", "surface_solar_radiation_w_m2")
VAR_LABELS = {
    "t2m_c": "2 m temperature (°C)",
    "wind_speed_10m": "10 m wind speed (m/s)",
    "surface_solar_radiation_w_m2": "Downward shortwave (W/m²)",
}

era5_june = Path("data/raw/era5_2019_de_bbox5deg_06.nc")
if not era5_june.is_file():
    msg = f"Missing {era5_june} — run py scripts/download_era5.py first."
    raise FileNotFoundError(msg)

In [ ]:
# ERA5 reanalysis at the plant point (hourly truth for the week)
era5 = ERA5Loader(tz=TZ).load(LAT, LON, WEEK_START, WEEK_END, tz=TZ)
era5.index.name = "valid_time"
era5 = era5.reset_index()
print(f"ERA5: {len(era5)} hourly rows  {era5['valid_time'].min()} → {era5['valid_time'].max()}")
era5[["valid_time", *COMPARE_VARS]].head()

In [ ]:
# GFS forecasts — one init every 12 h (00 & 12 UTC), four lead times each
inits = pd.date_range(WEEK_START, WEEK_END, freq="12h", tz=TZ)
print(f"Fetching {len(inits)} init cycles × {len(LEADS)} leads via Herbie …")

nwp_parts: list[pd.DataFrame] = []
for init in inits:
    try:
        part = extract_nwp_forecast_point(
            init,
            latitude=LAT,
            longitude=LON,
            fxx=LEADS,
            tz=TZ,
            verbose=False,
        )
        nwp_parts.append(part)
        print(f"  ✓ {init}")
    except FileNotFoundError as exc:
        print(f"  ✗ {init}: {exc}")

if not nwp_parts:
    raise RuntimeError("No GFS cycles retrieved — check network / init dates.")

nwp = pd.concat(nwp_parts)
nwp = nwp.reset_index()
print(f"NWP: {len(nwp)} forecast rows across {nwp['init_time'].nunique()} inits")
nwp.head()

In [ ]:
# Align NWP valid times with ERA5 truth
era5_slim = era5[["valid_time", *COMPARE_VARS]].rename(
    columns={col: f"{col}_era5" for col in COMPARE_VARS}
)
nwp_slim = nwp[["valid_time", "init_time", "lead_hours", *COMPARE_VARS]].rename(
    columns={col: f"{col}_nwp" for col in COMPARE_VARS}
)
merged = nwp_slim.merge(era5_slim, on="valid_time", how="inner")
print(f"Matched pairs: {len(merged)} (inner join on valid_time)")
merged.head()

In [ ]:
# --- Plot: ERA5 truth vs NWP forecasts for one week ---
fig, axes = plt.subplots(len(COMPARE_VARS), 1, figsize=(11, 8), sharex=True)
lead_colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(LEADS)))

for ax, var in zip(axes, COMPARE_VARS, strict=True):
    ax.plot(
        era5["valid_time"],
        era5[var],
        color="black",
        lw=1.5,
        label="ERA5 (truth)",
        zorder=3,
    )
    for lead, color in zip(LEADS, lead_colors, strict=True):
        sub = merged.loc[merged["lead_hours"] == lead]
        ax.scatter(
            sub["valid_time"],
            sub[f"{var}_nwp"],
            s=18,
            alpha=0.75,
            color=color,
            label=f"GFS f{lead:02d}",
            zorder=2,
        )
    ax.set_ylabel(VAR_LABELS[var])
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", ncol=2, fontsize=8)

axes[0].set_title(f"ERA5 vs GFS at ({LAT}°N, {LON}°E) — {WEEK_START} to {WEEK_END} (UTC)")
axes[-1].set_xlabel("Valid time (UTC)")
plt.tight_layout()
plt.show()

In [ ]:
# --- Lead-time MAE: NWP vs ERA5 (weather skill baseline) ---
mae_rows: list[dict[str, object]] = []
for lead in LEADS:
    sub = merged.loc[merged["lead_hours"] == lead]
    for var in COMPARE_VARS:
        err = (sub[f"{var}_nwp"] - sub[f"{var}_era5"]).abs()
        mae_rows.append(
            {
                "lead_hours": lead,
                "variable": var,
                "mae": float(err.mean()) if len(err) else np.nan,
                "n": len(err),
            }
        )

mae = pd.DataFrame(mae_rows)
mae_pivot = mae.pivot(index="lead_hours", columns="variable", values="mae")
mae_pivot = mae_pivot.reindex(LEADS)
mae_pivot.columns = [VAR_LABELS[c] for c in mae_pivot.columns]
print("Mean absolute error — GFS forecast vs ERA5 truth")
print(mae_pivot.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(LEADS))
width = 0.25
for i, var in enumerate(COMPARE_VARS):
    vals = mae.loc[mae["variable"] == var, "mae"].values
    ax.bar(x + i * width, vals, width=width, label=VAR_LABELS[var])
ax.set_xticks(x + width)
ax.set_xticklabels([f"f{lead:02d}" for lead in LEADS])
ax.set_xlabel("Forecast lead time")
ax.set_ylabel("MAE vs ERA5")
ax.set_title("Weather skill baseline — NWP error grows with lead time")
ax.legend(loc="upper left", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

**Notes**

- ERA5 is **reanalysis** (best estimate of past weather); GFS is an **operational forecast**. MAE here measures NWP weather error, not generation forecast skill.
- ERA5 `surface_solar_radiation_w_m2` is an hourly mean flux from accumulated J/m²; GFS `DSWRF` is an instantaneous flux — expect some bias in the solar comparison.
- Increase init frequency (e.g. `freq="6h"`) for smoother MAE estimates at the cost of longer download time.